<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: A keypad menu first

**[A keypad menu first](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/)** · Speech AI: Transcribe, Understand and Speak · Module 7, lesson 3 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read keypad tones (DTMF) from phone audio with the Goertzel algorithm, build a small keypad menu as a set of states, run the recorded test calls through it, and add the way to a person.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | What an IVR is and its states, from [What an IVR is, and why companies still use one](https://learning.nextia-ai.com/courses/speech/m07/what-an-ivr-is/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M07-L03-a-keypad-menu-first/a-keypad-menu-first-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/ivr/calls.jsonl": "5ee1fbad92c79b73fa908076325f2ce6d4030ed12439dacc26b6dcd9bc00d828",
    "data/orders.json": "74f313e9e6c6eac155c655f2c4ed5016ae8aa6076c38ca88edf7c09ed901aeda",
    "data/ivr/I-017/01.wav": "ad1dbd8a02463c4ec63979615e1183423543d2a4561942804dc8b1c096b6cb55",
    "data/ivr/I-017/02.wav": "90d94a27527f58a0d49a2d3017ce3ec19cdb6c0d7aa99edc22a0a4872a7673c4",
    "data/ivr/I-017/03.wav": "de9a9b4e9f7fe3e09b9e01b1085b8f2235d4c970ffc4b731d861391befbc4cb4",
    "data/ivr/I-017/04.wav": "37d2d128027b30442b0babfc52336d5a133abbb2e623c0af7cd287257da3878f",
    "data/ivr/I-019/01.wav": "8419242137ad05c4b667c59debbef4421e3ed5d53b41314b5b36c0932477adbe",
    "data/ivr/I-019/02.wav": "b8e4ce15840faf2020878d8ae0420a98871085a37b44fca01b78ce2c3fdd37a7",
    "data/ivr/I-019/03.wav": "4cb73a014422749ee723fb6fdb40ddf689fe8de47af4e5ea134df35dd5ae71d1",
    "data/ivr/I-019/04.wav": "3cfc3a93b604038912f91cc0a1bf9d02a1e61c0596c59a5f8dd5bbd7e5402c5f",
    "data/ivr/I-021/01.wav": "863d2ce65cb82f91206b98c3d0c079f26437a7656e238b0cd1ea7d4941770cb8",
    "data/ivr/I-021/02.wav": "239ef05184f7120bdc42e38af5f6c85d2762bf85766ce24ec80419bf6ea2389c",
    "data/ivr/I-021/03.wav": "4eff1303cda6b3029a9aa776fac1698843de623da63a557b6dcb88247794cd39",
    "data/ivr/I-021/04.wav": "b50552d15d0edd2f231c7978e0df028a79ec40f4cf021b8cf4ae91542ba8e71b",
    "data/ivr/I-022/01.wav": "98ee00832576f6b7bc724c232159544986ce2ee226ed59777a5609307cce8b94",
    "data/ivr/I-022/02.wav": "010be9a5b890ed8cc4384592256e85da04e2b3970ea2c6bf9272f548da786c52",
    "data/ivr/I-022/03.wav": "4fc13adb7d33a8fc801fdcc09def1f37d8a9075c0fc5d4913cc216c69c83b874",
    "data/ivr/I-022/04.wav": "2f96c61f72fd14b6f001a823590f3064c74c0dc41eb0e89d558cca57fe07f739",
    "data/ivr/I-033/01.wav": "23ef3a2b68deb1d3b027814d5bc3abc9263aa380f1cbb351dc05efbc113c473e",
    "data/ivr/I-033/02.wav": "bd0ef941f22a3ef424d7630f0f80bd859ce7c987fe5831cf3a79cb0915b61e7f",
    "data/ivr/I-062/01.wav": "ffe28dcd6c82e33d09999aaccc0b9bcb7e09c73111116a84f7a4924b86c34f46",
    "data/ivr/I-062/02.wav": "5ab322585d020957db13afa87ee7567ea292a00b590852dd499d8b0167ef8037",
    "data/ivr/I-062/03.wav": "ed5aa6636811cc040c22c07db24730ca7fbbe55868a3d3da612f6ac0c7d8b29b",
    "data/ivr/I-062/04.wav": "9b642da34c741f9743bd8c8d57b417019dc6e2185bd56378620ea6b41fb51108",
    "data/ivr/I-063/01.wav": "dd6d7b40872d0ba279fd5c0e4373123dd56cb836d24c49566d0e5883816287a7",
    "data/ivr/I-063/02.wav": "4697a83e7f00d07ae09410bf5a3540b8f1bcb7a02ab7ff5f20f61097c27928ba",
    "data/ivr/I-063/03.wav": "31984b85ab0b589a8fb500d47702c73815d334a9d77633c320949634733ef32f",
    "data/ivr/I-063/04.wav": "59369f192d2dc5b38996043d1ef45f5918d524db071a76ccbe5f86979bdb11fd",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A key is two tones

Amira builds Larkfield's out-of-hours line. She starts with the keypad, because it is predictable and cheap: "press 1 for orders".

When a caller presses a key, the phone plays two tones at the same time: one low and one high. This is **DTMF** (dual-tone multi-frequency). Key 5 is 770 Hz plus 1336 Hz.

| | 1209 Hz | 1336 Hz | 1477 Hz |
|---|---|---|---|
| **697 Hz** | 1 | 2 | 3 |
| **770 Hz** | 4 | 5 | 6 |
| **852 Hz** | 7 | 8 | 9 |
| **941 Hz** | * | 0 | # |

To find a key, you only need the strength of these 7 frequencies. The **Goertzel algorithm** measures one frequency with one short loop, without a full spectrum. Run the next cell: it makes the sound of key 5 and measures the 7 frequencies in its first 20 ms.

> **Predict.** Which two of the 7 numbers will be large?

In [ ]:
LOW = [697, 770, 852, 941]
HIGH = [1209, 1336, 1477]
KEYS = [["1", "2", "3"], ["4", "5", "6"], ["7", "8", "9"], ["*", "0", "#"]]

def goertzel(frame, sr, freq):
    """The strength of one frequency in a frame (the Goertzel algorithm: one loop, no full spectrum)."""
    k = 2 * np.cos(2 * np.pi * freq / sr)
    s1 = s2 = 0.0
    for x in frame:
        s1, s2 = x + k * s1 - s2, s1
    return s1 * s1 + s2 * s2 - k * s1 * s2

def tones(keys, sr=8000, on=0.1, off=0.1):
    """The sound of keys pressed on a phone: two sine waves for each key."""
    out = []
    for key in keys:
        r = next(i for i, row in enumerate(KEYS) if key in row)
        c = KEYS[r].index(key)
        t = np.arange(int(sr * on)) / sr
        out += [0.3 * (np.sin(2 * np.pi * LOW[r] * t) + np.sin(2 * np.pi * HIGH[c] * t)), np.zeros(int(sr * off))]
    return np.concatenate(out).astype(np.float32)

frame = tones("5")[:160]       # the first 20 ms of the key 5
for f in LOW + HIGH:
    print(f"{f:5} Hz  {goertzel(frame, 8000, f):10.1f}")

> **Check.** 770 Hz and 1336 Hz are much larger than the other five: key 5.

## 2. From frames to keys

The detector cuts the audio into 20 ms frames. A frame holds a key when two tones (one low, one high) carry most of its energy, and each is at least 4 times stronger than the next one in its group. A key counts once, when it is in two frames in a row: a short click or a sound in speech does not count.

In [ ]:
def key_in(frame, sr=8000):
    """The key whose two tones fill this frame, or None."""
    frame = np.asarray(frame, dtype=np.float64)
    energy = float(np.dot(frame, frame)) + 1e-12
    p = np.array([goertzel(frame, sr, f) for f in LOW + HIGH])
    lows, highs = p[:4], p[4:]
    i, j = int(np.argmax(lows)), int(np.argmax(highs))
    share = (lows[i] + highs[j]) * 2 / len(frame) / energy   # the share of the frame's energy in the two tones
    if share < 0.6 or lows[i] < 4 * np.sort(lows)[-2] or highs[j] < 4 * np.sort(highs)[-2]:
        return None
    return KEYS[i][j]

def read_keys(x, sr=8000, frame_s=0.02):
    """All the keys in a recording: a key counts once, when it is in two frames in a row."""
    n, found, last, count, down = int(sr * frame_s), "", None, 0, False
    for start in range(0, len(x) - n + 1, n):
        k = key_in(x[start:start + n], sr)
        count = count + 1 if (k and k == last) else (1 if k else 0)
        last = k
        if k and count == 2 and not down:
            found, down = found + k, True
        if not k:
            down = False
    return found

print(read_keys(tones("586032")))

> **Check.** You should see `586032`: six keys, each read once.

## 3. A recorded test call

The course has 60 recorded IVR test calls: the caller's side only, one audio file for each turn (speech, keys or silence), with the expected result. The keys went through a phone line, like real ones. Run the next cell: it reads the keys of the first keypad-only call.

In [ ]:
ivr_calls = {}
for line in open("data/ivr/calls.jsonl", encoding="utf-8"):
    row = json.loads(line)
    ivr_calls[row["call_id"]] = row
keypad_calls = [c for c in ivr_calls.values() if all("keys" in i for i in c["inputs"])]
print(len(ivr_calls), "IVR test calls;", len(keypad_calls), "use only the keypad:", [c["call_id"] for c in keypad_calls])
one = keypad_calls[0]
for inp in one["inputs"]:
    x, sr = read_audio("data/ivr/" + inp["file"], sr=8000)
    print(inp["file"], "pressed:", inp["keys"], "  read from the audio:", read_keys(x, sr))

> **Check.** 7 calls use only the keypad. For I-017, the keys read from the audio are the keys that were pressed.

## 4. The keypad menu as states

The menu is a small **state machine**. In each state it listens for keys, then it moves on, asks again, or hands over:

- `language`: 1 English, 2 French;
- `menu`: 1 orders, 2 opening hours, 3 returns;
- `number`: the 6-digit order number, checked with its check digit and in the order system;
- `confirm`: the number read back; 1 yes, 2 no;
- `done`: with an outcome (`status_given`, `hours_given`, `return_started` or `handover`).

A wrong key or no key is a **miss**: the menu asks again once, in a different way; after two misses a person takes over. Run the next cell. It defines the menu.

In [ ]:
orders = {o["order"]: o for o in json.load(open("data/orders.json"))}
KEY_INTENT = {"1": "order_status", "2": "opening_hours", "3": "start_return"}

def luhn_ok(number):
    """Larkfield order numbers end with a check digit (Luhn), so most typing mistakes are caught."""
    total = 0
    for i, ch in enumerate(reversed(number)):
        d = int(ch) * (2 if i % 2 else 1)
        total += d - 9 if d > 9 else d
    return total % 10 == 0

class KeypadMenu:
    """States: language -> menu -> number -> confirm -> done. Two misses in a state: a person takes over."""
    def __init__(self, orders):
        self.orders, self.state, self.language = orders, "language", None
        self.intent = self.order = self.outcome = self.handoff_reason = None
        self.misses, self.said = 0, []

    def say(self, text):
        self.said.append(text)

    def goto(self, state, prompt):
        self.state, self.misses = state, 0
        self.say(prompt)

    def end(self, outcome, prompt):
        self.outcome, self.state = outcome, "done"
        self.say(prompt)

    def transfer(self, reason):
        self.outcome, self.handoff_reason, self.state = "handover", reason, "done"
        self.say("Please hold: I am putting you through to a person.")

    def miss(self, prompt):
        """A wrong key or no key: ask again once, in a different way, then hand over."""
        self.misses += 1
        if self.misses >= 2:
            self.transfer("no_match")
        else:
            self.say(prompt)

    def step(self, keys):
        if self.state != "done":
            getattr(self, "on_" + self.state)(keys)

    def on_language(self, keys):
        self.language = {"1": "en", "2": "fr"}.get(keys[:1])
        if self.language:
            self.goto("menu", "For your orders, press 1. For opening hours, press 2. To return an item, press 3.")
        else:
            self.miss("Sorry. For English, press 1. Pour le français, faites le 2.")

    def on_menu(self, keys):
        self.intent = KEY_INTENT.get(keys[:1])
        if self.intent is None:
            self.miss("Sorry, I did not get that. Press 1 for orders, 2 for opening hours, or 3 for returns.")
        elif self.intent == "opening_hours":
            self.end("hours_given", "We are open Monday to Friday, 8 a.m. to 8 p.m., and Saturday, 9 to 5.")
        else:
            self.goto("number", "Please type your 6-digit order number.")

    def on_number(self, keys):
        if len(keys) == 6 and luhn_ok(keys) and keys in self.orders:
            self.order = keys
            self.goto("confirm", f"Order {keys[:3]} {keys[3:]}. If this is right, press 1. If not, press 2.")
        else:
            self.miss("I could not find that order. Please type the 6 digits again.")

    def on_confirm(self, keys):
        if keys[:1] == "1":
            o = self.orders[self.order]
            if self.intent == "start_return":
                if o["returnable"]:
                    self.end("return_started", f"I have started a return for your {o['product']}.")
                else:
                    self.transfer("not_returnable")
            else:
                self.end("status_given", f"Your {o['product']} is {o['status'].replace('_', ' ')}.")
        elif keys[:1] == "2":
            self.order = None
            self.miss("Please type your 6-digit order number again.")
            if self.state != "done":
                self.state = "number"
        else:
            self.miss("Press 1 if the order number is right, or 2 if it is not.")

## 5. Run the test calls

Each test call has an expected language, intent and outcome (and order number, or the reason for the handover). The next cell plays every keypad-only call into the menu and compares.

> **Predict.** Some test calls press 0 to talk to a person. Our menu has no 0. What happens to these calls?

In [ ]:
def replay(call, menu_class):
    """Play a recorded test call into the menu: read the keys from each audio file, step the menu."""
    menu = menu_class(orders)
    for inp in call["inputs"]:
        x, sr = read_audio("data/ivr/" + inp["file"], sr=8000)
        menu.step(read_keys(x, sr))
    got = {"language": menu.language, "intent": menu.intent, "outcome": menu.outcome}
    if "order" in call["expected"]:
        got["order"] = menu.order
    if "handoff_reason" in call["expected"]:
        got["handoff_reason"] = menu.handoff_reason
    want = {k: call["expected"][k] for k in got}
    return got == want, got, menu

def run_tests(menu_class):
    results = {}
    for call in keypad_calls:
        ok, got, _ = replay(call, menu_class)
        results[call["call_id"]] = ok
        print(f'{call["call_id"]} {call["kind"]:14} {"pass" if ok else "FAIL"}  {got}')
    print(f"{sum(results.values())} of {len(results)} passed")
    return results

base_results = run_tests(KeypadMenu)

> **Check.** 5 of 7 pass. The calls that fail press 0 (I-062, I-063): the menu treats 0 as a wrong key, asks again, and hands over with `no_match` after two misses. The caller gets a person, but only after two "Sorry" messages, and the call record gives the wrong reason.

## 6. Change one thing: short key presses

Some phones send very short tones.

> **Predict.** With tones of 20 ms instead of 100 ms, how many of the six keys does the detector read?

In [ ]:
for on in (0.1, 0.04, 0.02):
    print(f"tones of {on * 1000:.0f} ms: read", repr(read_keys(tones("586032", on=on, off=on))), "for '586032'")

> **Check.** At 40 ms each key still fills two 20 ms frames, so all six are read. At 20 ms a key never fills two whole frames in a row, so no key is read. The standard asks for tones of at least 40 ms; the two-frame rule is the reason.

## 7. Failure case: the wrong sample rate

The detector's frequencies depend on the sample rate. If you read the audio at 16 kHz and tell the detector 8 kHz, every frequency looks twice as low as it is.

In [ ]:
x16, _ = read_audio("data/ivr/" + one["inputs"][2]["file"], sr=16000)   # 16 kHz samples
print("read as if 8 kHz:", repr(read_keys(x16, sr=8000)), "  read at the right rate:", repr(read_keys(x16, sr=16000)))

> **Check.** At the wrong rate, the detector reads nothing (or wrong keys), with no error message. **Fix:** pass the real sample rate, and test the detector with `tones()` at the rate of your phone line.

## 8. Your turn: 0 for a person

> **Your turn.** Callers expect 0 to reach a person in every state. Complete `KeypadMenuWithPerson.step`: when the first key is `"0"` and the call is not done, set the intent to `"agent"` (if there is no intent yet) and transfer with the reason `"caller_asked"`; do not call the parent's `step` then. For all other keys, call `super().step(keys)`. Then run the tests again.

In [ ]:
class KeypadMenuWithPerson(KeypadMenu):
    def step(self, keys):
        if keys[:1] == "0" and self.state != "done":
            self.intent = self.intent or "agent"
            self.transfer("caller_asked")
            return
        super().step(keys)

In [ ]:
person_results = run_tests(KeypadMenuWithPerson)

In [ ]:
expect_hash('the test results', person_results, 'fee6677a8af631cb')

> **Check.** Every keypad-only test call passes. A real IVR also needs the speech path, which the next lesson adds: the keypad stays as the fallback when speech fails twice.

## Next

You read keypad tones from phone audio, built a keypad menu as states, tested it with recorded calls, and added the way to a person. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Let callers say what they need](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m07/a-keypad-menu-first/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.